# W0에서 pin_all / fixed_ttl 돌려보기 (5주차)

5주차 실험 발표 범위: 정책 2개(pin_all, fixed_ttl), W0 기준 운영, 부하 1x, 산출은 메모리 점유 곡선 + smoke test.
트레이스는 `kvbench.workload`(configs/smoke.json), 재생은 `kvbench.simulate`. 결과는 results/w0_baseline/ (gitignore).
프로젝트 루트에서 `jupyter lab` 을 띄우고 열면 됨. pandas, matplotlib 필요.

In [ ]:
import sys, os, time, dataclasses
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

# 프로젝트 루트로 이동 (notebooks/ 에서 열었다고 가정)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

from kvbench.workload import load_config, generate, write_trace
from kvbench.system import load_system_config, load_policy_params
from kvbench.simulate import simulate, write_results

policies = ["pin_all", "fixed_ttl"]
seeds = [0, 1, 2, 3, 4]
out = Path("results/w0_baseline")
(out / "traces").mkdir(parents=True, exist_ok=True)
(out / "figs").mkdir(exist_ok=True)
print(ROOT)

## 실행 환경
어디서 돌렸는지 남겨두기 (서버 / 커밋)

In [ ]:
!hostname
!git log -1 --oneline
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 시스템 파라미터
configs/system_1080ti_llama3b.json 그대로. prefill_tokens_per_s, decode_tokens_per_s 는 자리표시자 -> 실측하면 json 만 바꾸고 다시 돌리면 됨.
두 정책 다 오프로드가 없어서 pcie 값은 안 씀.

In [ ]:
system = load_system_config("configs/system_1080ti_llama3b.json")
pd.Series(dataclasses.asdict(system))

In [ ]:
print("total blocks:", system.total_blocks, "/ block =", system.block_bytes / 2**20, "MiB")
sess4k = 4096 * system.kv_bytes_per_token
print("4K 세션 =", sess4k / 2**20, "MiB ->", system.gpu_kv_pool_bytes // sess4k, "개면 포화")

## 트레이스 생성
configs/smoke.json 의 W0, seed 0~4. normal 은 단일 턴 요청, stress 가 도구를 쓰는 다중 턴 세션.

In [ ]:
cfg = load_config("configs/smoke.json")
traces = {}
for seed in seeds:
    rows = generate(cfg, "W0", 1, seed)
    path = out / "traces" / f"w0_s1_seed{seed}.jsonl"
    summary = write_trace(path, rows, cfg, "W0", 1, seed)
    traces[seed] = path
    print(path.name, summary)

In [ ]:
df = pd.concat([pd.read_json(p, lines=True) for p in traces.values()])
stress = df[df.request_class == "stress"]
wait = stress[stress.tool_duration_ms > 0].tool_duration_ms / 1000
print(wait.describe(percentiles=[.5, .9]).round(2))
held = (stress.context_tokens + stress.output_tokens).groupby([stress.seed, stress.session_id]).max()
print("\nstress 세션 최대 보유 토큰: median", held.median(), "/ max", held.max(),
      "->", round(held.max() * system.kv_bytes_per_token / 2**20), "MiB")
print("warmup_ms:", cfg.warmup_ms, "/ duration_ms:", cfg.duration_ms)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].hist(wait, bins=30); ax[0].set_xlabel("tool wait (s)")
ax[1].hist(held, bins=30); ax[1].set_xlabel("tokens held in KV (stress, max per session)")
plt.tight_layout()

## 정책 파라미터

In [ ]:
for p in policies:
    print(p, load_policy_params("configs/policies.json", p))

ttl = load_policy_params("configs/policies.json", "fixed_ttl")["ttl_ms"] / 1000
print(f"\nTTL {ttl:.0f}s 보다 긴 대기 비율: {(wait > ttl).mean() * 100:.1f}%")   # 작으면 fixed_ttl 이 pin_all 이랑 거의 같아짐

## smoke test
seed 0 하나로 두 정책이 끝까지 도는지 먼저 확인. CLI 로는 `python -m kvbench simulate --trace ... --policy pin_all --output ...` 와 같음.

In [ ]:
def run(seed, policy):
    sim, s = simulate(traces[seed], policy, system, load_policy_params("configs/policies.json", policy))
    return write_results(sim, s, out / "runs" / f"w0_s1_seed{seed}_{policy}", overwrite=True)   # run_id 가 붙어서 돌아옴

for p in policies:
    s = run(0, p)
    print(f"{p:10s} sessions {s['sessions']}  completed {s['completed_jobs']}  rejected {s['rejected_jobs']}  "
          f"end {s['end_ms'] / 1000:.0f}s  p95 ttft {s['normal_p95_ttft_ms']:.0f} ms  peak {s['peak_kv_occupancy_pct']:.1f}%")

## seed 5개 실행

In [ ]:
res = []
t0 = time.time()
for p in policies:
    for seed in seeds:
        s = run(seed, p)
        res.append(s)
        print(f"{s['run_id']:22s} p95 ttft {s['normal_p95_ttft_ms']:8.0f} ms  reject {s['normal_reject_rate_pct']:4.1f}%  "
              f"peak {s['peak_kv_occupancy_pct']:5.1f}%  evict {s['eviction_count']:3d}  recompute {s['recomputed_tokens']:6d}")
res = pd.DataFrame(res)
res.to_csv(out / "summary.csv", index=False)
print(len(res), "runs", round(time.time() - t0, 1), "s")

## 지표 표 (seed 평균)
SLO 는 reject <= 10%, 점유율 <= 90% 만 봄. TTFT 3배 기준은 1x 자체가 baseline 이라 여기서는 판정 안 함 (여기 p95 가 나중에 기준값).

In [ ]:
cols = ["normal_p95_ttft_ms", "normal_reject_rate_pct", "peak_kv_occupancy_pct", "eviction_count", "recomputed_tokens"]
mean = res.groupby("policy")[cols].mean().loc[policies].round(1)
std = res.groupby("policy")[cols].std().loc[policies].round(1)
mean["reject_ok"] = mean.normal_reject_rate_pct <= 10
mean["occ_ok"] = mean.peak_kv_occupancy_pct <= 90
mean

In [ ]:
std

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(9, 3))
for a, c in zip(ax, cols[:3]):
    a.bar(policies, mean[c], yerr=std[c], capsize=3, width=0.5)
    a.set_title(c, fontsize=10); a.set_ylim(bottom=0)
ax[1].axhline(10, color="gray", ls=":"); ax[2].axhline(90, color="gray", ls=":")
plt.tight_layout()
plt.savefig(out / "figs/w0_metrics.png", dpi=150)

## 메모리 점유 곡선
seed 0 한 트레이스를 정책만 바꿔서. 회색은 warmup (지표에서 제외), 점선 90%.
두 곡선이 거의 겹치면 TTL 보다 긴 대기가 거의 없다는 뜻 (fixed_ttl 이 삭제할 일이 별로 없음).

In [ ]:
def occ(run_id):
    o = pd.read_csv(out / "runs" / run_id / "occupancy.csv")
    return o.time_ms / 1000, o.used_blocks / o.total_blocks * 100

plt.figure(figsize=(11, 3.5))
t, y = occ("w0_s1_seed0_pin_all")
plt.step(t, y, where="post", label="pin_all", lw=2)
t, y = occ("w0_s1_seed0_fixed_ttl")
plt.step(t, y, where="post", label="fixed_ttl", lw=1, ls="--")
plt.axvspan(0, cfg.warmup_ms / 1000, color="0.9")
plt.axhline(90, color="gray", ls=":")
plt.xlabel("time (s)"); plt.ylabel("KV pool used (%)"); plt.title("W0 seed 0"); plt.legend()
plt.tight_layout()
plt.savefig(out / "figs/w0_occupancy.png", dpi=150)

In [ ]:
# seed 별로도 한번
fig, axes = plt.subplots(2, 1, figsize=(11, 4.5), sharex=True)
for a, p in zip(axes, policies):
    for seed in seeds:
        t, y = occ(f"w0_s1_seed{seed}_{p}")
        a.step(t, y, where="post", lw=0.8)
    a.set_title(p, loc="left", fontsize=10); a.axhline(90, color="gray", ls=":")
plt.tight_layout()
plt.savefig(out / "figs/w0_occupancy_seeds.png", dpi=150)

## 정책이 의도대로 동작했는지 (events.csv)
- pin_all: EVICT 0 (idle 캐시 절대 안 버림)
- fixed_ttl: EVICT 는 전부 TTL_EXPIRE 뒤, idle 시작에서 정확히 ttl 뒤

In [ ]:
for seed in seeds:
    for p in policies:
        ev = pd.read_csv(out / "runs" / f"w0_s1_seed{seed}_{p}" / "events.csv")
        n_idle = (ev.event == "TOOL_START").sum()
        n_evict = (ev.event == "EVICT").sum()
        n_ttl = (ev.event == "TTL_EXPIRE").sum()
        if p == "pin_all":
            ok = n_evict == 0
        else:
            e = ev[ev.event == "EVICT"].merge(ev[ev.event == "TOOL_START"][["session_id", "turn_id", "time_ms"]],
                                              on=["session_id", "turn_id"], suffixes=("", "_idle"))
            gap = e.time_ms - e.time_ms_idle
            ok = n_evict == n_ttl and np.allclose(gap, ttl * 1000)
        print(f"seed{seed} {p:10s} idle={n_idle:4d} evict={n_evict:3d} ttl_expire={n_ttl:3d} -> {'ok' if ok else 'FAIL'}")

## W0 도착률 확인
1x 에서 pin_all 이 이미 90% 넘으면 W0 가 기준 워크로드 역할을 못 함. stress 도착률을 바꿔가면서 pin_all 만 돌려봄 (smoke.json 은 안 건드림).

In [ ]:
from kvbench.engine import Simulator
from kvbench.metrics import summarize
from kvbench.policies import make_policy

rows_ = []
for rate in [0.1, 0.2, 0.3, 0.4, 0.5]:
    c = dataclasses.replace(cfg, stress_arrival_rate_per_s=rate)
    for seed in seeds:
        m = summarize(Simulator(generate(c, "W0", 1, seed), system, make_policy("pin_all"), keep_events=False).run(), c.warmup_ms)
        rows_.append((rate, m["peak_kv_occupancy_pct"], m["normal_reject_rate_pct"], m["normal_p95_ttft_ms"]))
sw = pd.DataFrame(rows_, columns=["stress_rate", "peak", "reject", "p95"])
sw.groupby("stress_rate").agg(["mean", "max"]).round(1)

## 메모
- smoke.json 은 동작 확인용이라 60초짜리. 발표용으로는 duration 을 늘리고 도착률을 위 sweep 보고 정해야 함
- fixed_ttl 30s 는 smoke.json 의 대기 분포(1~5초) 대비 너무 길어서 pin_all 이랑 사실상 같음. TTL 을 대기 분포 기준(p50~p75)으로?
- prefill/decode 속도 실측 후 json 바꾸고 다시 돌리기. 그 전엔 TTFT 절대값 말고 정책 간 비교 + 점유율만
- 그림은 results/w0_baseline/figs/